# Example 1: Symbolic Regressor

In [ ]:
import sys
import os
from pathlib import Path

from pyparsing import results

# Add the parent folder (my_project) to Python's search path
sys.path.append(str(Path.cwd().parent))

%matplotlib inline
from gplearn.genetic_pmm import SymbolicRegressor
from sklearn.utils.random import check_random_state
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import r2_score
import math
from joblib import dump, load

os.environ["PATH"] += os.pathsep + r'C:\Program Files\Graphviz\bin'

In [ ]:
def get_benchmark_dataset(name, n_train=100, n_test=100, random_state=0):
    """
    Generates train and test datasets for standard Genetic Programming benchmarks.

    Parameters:
        name (str): Benchmark problem ('E1', 'E2', 'M1', 'M2', 'H1', 'H2')
        n_train (int): Number of training samples
        n_test (int): Number of testing samples
        random_state (int/RandomState): Seed for reproducibility

    Returns:
        X_train, y_train, X_test, y_test
    """
    rng = check_random_state(random_state)
    name = name.upper().replace("-", "").replace("_", "")

    def _generate_data(n_samples):

        # 1. Custom Domain Handling
        if name == 'NGUYEN7':
            # Sampling x from [0, 2] to prevent log(x + 1) domain errors (x <= -1)
            X = rng.uniform(0, 2, n_samples * 5).reshape(-1, 5)
        elif name == 'KEIJZER6':
            # Sampling positive values [1, 100] for harmonic number calculations
            X = rng.uniform(1, 100, n_samples * 5).reshape(-1, 5)
        else:
            # Default standard domain [-5, 5]
            X = rng.uniform(-5, 5, n_samples * 5).reshape(-1, 5)

        x1, x2, x3, x4, x5 = X[:, 0], X[:, 1], X[:, 2], X[:, 3], X[:, 4]

        # Define Function

        # 1. Koza Benchmark Suite
        if name == 'KOZA1':
            y = x1**4 + x1**3 + x1**2 + x1
        elif name == 'KOZA2':
            y = x1**5 - 2 * (x1**3) + x1
        elif name == 'KOZA3':
            y = x1**6 - 2 * (x1**4) + x1**2

        # 2. Nguyen Benchmark Suite
        elif name == 'NGUYEN1':
            y = x1**3 + x1**2 + x1
        elif name == 'NGUYEN3':
            y = x1**5 + x1**4 + x1**3 + x1**2 + x1
        elif name == 'NGUYEN5':
            y = np.sin(x1**2) * np.cos(x1) - 1
        elif name == 'NGUYEN6':
            y = np.sin(x1) * np.sin(x1 + x1**2)
        elif name == 'NGUYEN7':
            y = np.log(x1 + 1) + np.log(x1**2 + 1)
        elif name == 'NGUYEN10':
            y = np.sin(x1) + np.sin(x2**2)

        # 3. Pagie Polynomial
        elif name == 'PAGIE1':
            y = (1 / (1 + x1**-4)) + (1 / (1 + x2**-4))

        # 4. Keijzer Suite
        elif name == 'KEIJZER6':
            # Harmonic sum: sum_{i=1..x1} (1 / i)
            y = np.array([sum(1.0 / i for i in range(1, int(val) + 1)) for val in x1])

        # 5. Vladislavleva Suite
        elif name == 'VLADISLAVLEVA1':
            y = np.exp(-(x1 - 1)**2) * np.cos(x2)

        else:
            raise ValueError(f"Unknown benchmark name: '{name}'. Choose from 'E1', 'E2', 'M1', 'M2', 'H1', 'H2'.")

        return X, y

    X_train, y_train = _generate_data(n_train)
    X_test, y_test = _generate_data(n_test)

    return X_train, y_train, X_test, y_test

In [ ]:
def get_feature_map(function_set, n_features):
    """Creates a dictionary mapping every function name and feature index to a vector slot."""
    feature_map = {}
    idx = 0

    # 1. Map all function names (e.g., 'add', 'sub', 'mul')
    for func in function_set:
        feature_map[func.name] = idx
        idx += 1

    # 2. Map all feature indices (0, 1, 2, ... n_features - 1)
    for feat_idx in range(n_features):
        feature_map[feat_idx] = idx
        idx += 1

    # 3. Map Constant Bins
    constant_bins = [
        'CONST_Q0_To_Q1',  # c < min + 0.25 * range
        'CONST_Q1_To_Q2',  # min + 0.25 * range <= c < min + 0.50 * range
        'CONST_Q2_To_Q3',  # min + 0.50 * range A<= c < min + 0.75 * range
        'CONST_Q3_To_Q4'   # c >= min + 0.75 * range
    ]

    for bin_name in constant_bins:
        feature_map[bin_name] = idx
        idx += 1

    return feature_map

In [ ]:
def plot_all_benchmarks_histogram(n_samples=10000, random_state=42, benchmarks = []):
    """
    Plots a 2x3 grid of histograms showing target distributions across all benchmark functions.
    """
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
    axes = axes.flatten()

    for idx, b_name in enumerate(benchmarks):
        X_tr, y_tr, _, _ = get_benchmark_dataset(b_name, n_train=n_samples, random_state=random_state)

        # Clip extreme asymptote values for H1/F16 (tan x) so outliers don't flatten the plot
        if b_name == 'H1':
            y_display = np.clip(y_tr, -100, 100)
            title_suffix = " (Clipped [-100, 100])"
        else:
            y_display = y_tr
            title_suffix = ""

        ax = axes[idx]
        sns.histplot(y_display, bins=50, kde=True, ax=ax, color='skyblue', edgecolor='black', alpha=0.7)

        # Summary stats in title
        ax.set_title(
            f"Benchmark {b_name}{title_suffix}\nMean: {np.mean(y_tr):.2f} | Std: {np.std(y_tr):.2f}",
            fontsize=11,
            fontweight='bold'
        )
        ax.set_xlabel("Target Response (y)")
        ax.set_ylabel("Count")

    plt.tight_layout()
    plt.show()

# Run combined plotting function
# plot_all_benchmarks_histogram(benchmarks=benchmarks)

In [ ]:
benchmarks = [
    'koza-1', 'koza-2','koza-3',
    # 'nguyen-1', 'nguyen-3', 'nguyen-5',
    # 'nguyen-6', 'nguyen-7', 'nguyen-10',
    # 'pagie-1', 'keijzer-6', 'vladislavleva-1',
]

total_run = len(benchmarks)
cols = 3
rows = math.ceil(total_run / cols)


In [ ]:
# Training Settings

def get_params(random_seed):
    return dict(
    population_size=500,
    generations=200,
    stopping_criteria=0.005,
    tournament_size= 20,
    p_crossover=0.7,
    p_subtree_mutation=0.1,
    p_hoist_mutation=0.05,
    p_point_mutation=0.1,
    max_samples=1.0,
    verbose=1,
    parsimony_coefficient=0.005,
    # parsimony_coefficient=0.01,
    random_state=random_seed,
    metric='rmse',
    function_set = ('add', 'sub', 'mul','div', 'sqrt', 'log',
                    'abs', 'neg', 'inv',
                    'max', 'min',
                    'sin', 'cos', 'tan'
                    ),

    prey_n_elites=1,
        predator_n_elites=1,

    predator_population_size= 50,
        catch_penalty= 0.00,
        catch_num= 1,
        catch_size= 20,

    prey_competitive_consts = 1,
        predator_competitive_consts = 1,

    data_record_diversity_distance = True,
        data_record_diversity_entropy = True,

)

seed_number = 4
is_skip = False

master_rng = np.random.default_rng(seed=42)
random_seeds = master_rng.integers(low=0, high=10000, size= seed_number).tolist()
print(f"Random Seeds = {random_seeds}")
# random_seeds = [0]

In [ ]:
results = []
is_break = False

for seed in random_seeds:
    print(f"\n==================== SEED {seed} ====================")

    for name in benchmarks:
        # 1. Load dataset using the current seed
        X_train, y_train, X_test, y_test = get_benchmark_dataset(name, random_state=seed)

        # 2. Instantiate GP with current random_state
        est_gp = SymbolicRegressor(**get_params(seed))
        print(f"\n====================================================================================")
        print(f"Benchmark Running = {name} with Seed {seed}")

        # 3. Train GP
        est_gp.fit(X_train, y_train)
        print(f"Formula: {str(est_gp._program)}")

        # 4. Predict and clean numerical issues
        y_pred = est_gp.predict(X_test)
        y_pred = np.nan_to_num(y_pred, nan=0.0, posinf=1e5, neginf=-1e5)

        # 5. Calculate Test R2
        r2 = r2_score(y_test, y_pred)
        r2_scores = [r2_score(y_train, program.execute(X_train)) for program in
                     est_gp.best_programs_per_gen]


        # 6. Record findings
        results.append({
            'Seed': seed,
            'Benchmark': name,
            'R2_Score': r2,
            'Formula': str(est_gp._program),

            # Data Per Generation
            'R2 Score Per Gen': r2_scores,

            'Prey Average Length': est_gp.run_details_['average_length'],
            'Prey Average Fitness': est_gp.run_details_['average_fitness'],
            'Prey Best Fitness': est_gp.run_details_['best_fitness'],
            'Prey Best Length': est_gp.run_details_['best_length'],

            'Predator Average Length': est_gp.run_details_['pred_average_length'],
            'Predator Average Fitness': est_gp.run_details_['pred_average_fitness'],
            'Predator Best Fitness': est_gp.run_details_['pred_best_fitness'],
            'Predator Best Length': est_gp.run_details_['pred_best_length'],

            'Prey Diversity Distance': est_gp.run_details_['prey_diversity_distance'],
            'Prey Diversity Entropy': est_gp.run_details_['prey_diversity_entropy'],
            'Predator Diversity Distance': est_gp.run_details_['pred_diversity_distance'],
            'Predator Diversity Entropy': est_gp.run_details_['pred_diversity_entropy'],

            'Predator Entropy History': est_gp.run_details_['pred_entropy_history'],
            'Prey Entropy History': est_gp.run_details_['prey_entropy_history'],
            'Feature Map': get_feature_map(est_gp._function_set, X_train.shape[1])
        })

        if r2 < 0.6 and is_skip:
            is_break = True
            break

    if (is_break and is_skip): break


In [ ]:
# Save the full results list to a compressed file
dump(results, 'Graphs/pmm_results.joblib', compress=3)
print("Results successfully saved to 'Graphs/pmm_results.joblib'")

# Load the exact object back into memory
# results = load('pmm_results.joblib')
# print(f"Loaded {len(results)} ppm records successfully.")